# RAG 기본 구조 이해하기 — 8단계 파이프라인 완전 정복

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- RAG(Retrieval-Augmented Generation)의 개념과 왜 필요한지 설명하기
- 사전작업(Pre-processing) 4단계와 실행(Runtime) 4단계를 구분하기
- LangChain LCEL로 8단계를 하나의 체인으로 연결하기

## 사전 지식

- 6-1~6-4 섹션의 DocumentLoader, Chunking, Embedding, VectorStore 개념
- LangChain의 `|` 연산자(LCEL 파이프라인) 기본 문법

---

```mermaid
flowchart LR
    subgraph PRE[사전작업 Pre-processing]
        D[문서 로드]:::process --> S[청크 분할]:::process
        S --> E[임베딩 생성]:::process
        E --> V[(벡터 DB 저장)]:::storage
    end
    subgraph RT[실행 Runtime]
        Q[사용자 질문]:::input --> R[Retriever<br/>관련 문서 검색]:::process
        V --> R
        R --> P[Prompt<br/>문서+질문 조합]:::process
        P --> L[LLM<br/>답변 생성]:::process
        L --> A[최종 답변]:::output
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## RAG란?

**RAG(Retrieval-Augmented Generation, 검색 증강 생성)**는 외부 문서에서 관련 정보를 검색하여 LLM의 답변 품질을 향상시키는 기법이에요.

| 일반 LLM | RAG |
|---------|-----|
| 학습 데이터만 참조 | 외부 문서 실시간 참조 |
| 환각(Hallucination) 가능성 | 근거 기반 답변 |
| 지식 업데이트 어려움 | 문서만 교체하면 최신화 |

## RAG 파이프라인 개요

RAG는 크게 **사전작업(Pre-processing)**과 **실행 단계(Runtime)**로 나뉩니다.

### 1. 사전작업 (Pre-processing) - 1~4단계

문서를 Vector DB에 준비하는 단계입니다:

```
문서 → 로드 → 분할 → 임베딩 → Vector DB 저장
```

- **1단계 - 문서 로드(Document Load)**: 문서 파일을 읽어옵니다
- **2단계 - 분할(Text Split)**: 문서를 작은 청크(Chunk)로 나눕니다
- **3단계 - 임베딩(Embedding)**: 각 청크를 벡터로 변환합니다
- **4단계 - 벡터DB 저장**: 임베딩된 청크를 데이터베이스에 저장합니다

### 2. 실행 단계 (Runtime) - 5~8단계

사용자 질문에 답변하는 단계입니다:

```
사용자 질문 → 검색 → 프롬프트 생성 → LLM 호출 → 답변 생성
```

- **5단계 - 검색기(Retriever)**: 질문과 관련된 문서를 검색합니다
- **6단계 - 프롬프트**: 검색된 문서와 질문을 조합합니다
- **7단계 - LLM**: 언어 모델을 정의합니다
- **8단계 - 체인(Chain)**: 전체 파이프라인을 연결합니다

## 환경 설정

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

## 실습 문서

이번 실습에서는 디지털 정책 관련 문서를 사용합니다.

- 파일명: `sample-rag-brief.pdf`
- 위치: `../6-1_DocumentLoaders/data/` 디렉토리

**학습 목표**: PDF 문서를 기반으로 질문에 답변할 수 있는 RAG 시스템 구축

## RAG 기본 파이프라인 구현

이제 8단계 파이프라인을 순서대로 구현하겠습니다.

### 필수 라이브러리 import

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### 1단계: 문서 로드 (Load Documents)

`PyMuPDFLoader`를 사용하여 PDF 문서를 로드합니다.

**주요 특징**:
- PDF 파일을 페이지별로 읽어옵니다
- 각 페이지는 별도의 `Document` 객체로 저장됩니다
- 메타데이터(페이지 번호, 파일명 등)도 함께 저장됩니다

In [ ]:
import os

# 데이터 파일 경로 확인
file_path = "../6-1_DocumentLoaders/data/sample-rag-brief.pdf"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}\n현재 디렉토리: {os.getcwd()}")

In [ ]:
# 단계 1: 문서 로드
loader = PyMuPDFLoader("../6-1_DocumentLoaders/data/sample-rag-brief.pdf")
docs = loader.load()

print(f"문서의 페이지 수: {len(docs)}")
print(f"첫 번째 페이지 미리보기: {docs[0].page_content[:200]}...")

특정 페이지의 전체 내용과 메타데이터를 확인해봅니다.

In [ ]:
# 2번째 페이지 내용 출력
if len(docs) < 2:
    print("⚠️ 로드된 문서가 2개 미만이라 두 번째 페이지를 표시할 수 없습니다.")
else:
    print("=" * 60)
    print("[2번째 페이지 내용]")
    print("=" * 60)
    print(docs[1].page_content)
    print("\n" + "=" * 60)
    print("[메타데이터]")
    print("=" * 60)
    print(docs[1].metadata)


### 2단계: 문서 분할 (Split Documents)

`RecursiveCharacterTextSplitter`를 사용하여 문서를 작은 청크로 분할합니다.

**분할이 필요한 이유**:
- LLM의 컨텍스트 윈도우 크기 제한
- 검색 정확도 향상 (작은 단위가 더 정확)
- 관련 정보만 선택적으로 전달 가능

**주요 파라미터**:
- `chunk_size`: 각 청크의 최대 문자 수 (일반적으로 500~1500)
- `chunk_overlap`: 청크 간 겹치는 문자 수 (문맥 유지를 위해 10~20% 권장)

In [ ]:
# 단계 2: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 각 청크의 최대 크기
    chunk_overlap=100  # 청크 간 겹치는 부분 (문맥 보존)
)
split_documents = text_splitter.split_documents(docs)

print(f"원본 문서 페이지 수: {len(docs)}")
print(f"분할된 청크 수: {len(split_documents)}")
print(f"\n첫 번째 청크 내용:\n{split_documents[0].page_content}")

### 3단계: 임베딩 (Embedding) 생성

`OpenAIEmbeddings`를 사용하여 텍스트를 벡터로 변환하는 임베딩 모델을 생성합니다.

**임베딩이란?**
- 텍스트를 고차원 벡터 공간의 점으로 표현하는 기술
- 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 배치됨
- 이를 통해 의미 기반 검색(Semantic Search)이 가능해짐

In [ ]:
# 단계 3: 임베딩 모델 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # OpenAI의 최신 임베딩 모델
)

print("임베딩 모델 생성 완료")
print(f"사용 모델: text-embedding-3-small")

### 4단계: 벡터 DB 생성 및 저장

`FAISS` 벡터스토어를 생성하고 분할된 문서를 임베딩하여 저장합니다.

**FAISS (Facebook AI Similarity Search)**:
- 메타(Meta)에서 개발한 고성능 벡터 검색 라이브러리
- 메모리 기반으로 빠른 검색 속도 제공
- 로컬 환경에서 간단히 사용 가능

In [ ]:
# 단계 4: 벡터스토어 생성 및 문서 저장
vectorstore = FAISS.from_documents(
    documents=split_documents,  # 분할된 문서
    embedding=embeddings  # 임베딩 모델
)

print(f"벡터스토어 생성 완료")
print(f"저장된 문서 청크 수: {len(split_documents)}")

벡터스토어가 제대로 작동하는지 간단한 검색으로 테스트해봅니다.

In [ ]:
# 벡터스토어 검색 테스트
test_query = "디지털 혁신"
search_results = vectorstore.similarity_search(test_query, k=2)

print(f"검색어: '{test_query}'")
print("=" * 60)
for i, doc in enumerate(search_results, 1):
    print(f"\n[검색 결과 {i}]")
    print(doc.page_content[:200] + "...")
    print(f"출처: 페이지 {doc.metadata.get('page', 'N/A')}")

### 5단계: 검색기 (Retriever) 생성

벡터스토어를 `Retriever`로 변환합니다.

**Retriever의 역할**:
- 사용자 질문을 입력받아 관련 문서를 자동으로 검색
- 검색 알고리즘과 파라미터를 캡슐화
- 체인에서 바로 사용 가능한 형태로 제공

In [ ]:
# ---------------------------------------------------
# 단계 5: 검색기(Retriever) 생성
# ---------------------------------------------------
# ============================================================
# TODO: 벡터스토어를 Retriever로 변환하세요
# 힌트: vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
# 예상 결과: "검색기(Retriever) 생성 완료" 출력
# ============================================================


검색기에 실제 질문을 넣어 어떤 문서가 검색되는지 확인해봅니다.

In [ ]:
# ---------------------------------------------------
# 검색기 테스트 — 질문으로 관련 문서 검색
# ---------------------------------------------------
# ============================================================
# TODO: retriever.invoke()로 질문을 검색하고 결과를 확인하세요
# 힌트: retriever.invoke("디지털 정부 혁신의 주요 목표는 무엇인가요?")
# 예상 결과: 검색된 문서 수와 각 문서의 미리보기 출력
# ============================================================


### 6단계: 프롬프트 생성 (Create Prompt)

RAG를 위한 프롬프트 템플릿을 생성합니다.

**프롬프트 구성 요소**:
- `context`: 검색된 문서 내용이 자동으로 입력됨
- `question`: 사용자의 질문이 입력됨
- 시스템 지시사항: LLM에게 역할과 답변 방식을 명확히 전달

In [ ]:
# ---------------------------------------------------
# 단계 6: RAG용 프롬프트 템플릿 정의
# ---------------------------------------------------
# ============================================================
# TODO: PromptTemplate.from_template()으로 RAG 프롬프트를 만드세요
# 힌트: {context}에 검색 문서, {question}에 사용자 질문이 들어가는 템플릿
# 예상 결과: "프롬프트 템플릿 생성 완료" 출력
# ============================================================


### 7단계: 언어 모델 (LLM) 생성

`ChatOpenAI` 모델을 생성합니다.

**모델 선택 가이드**:
- `gpt-4o-mini`: 빠르고 비용 효율적 (일반적인 QA에 적합)
- `gpt-4o`: 복잡한 추론이 필요한 경우
- `temperature=0`: 일관되고 결정적인 답변 생성 (RAG에서 권장)

In [ ]:
# 단계 7: 언어 모델 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",  # 빠르고 경제적인 모델
    temperature=0  # 결정적인 출력 (일관성 확보)
)

print("언어 모델 생성 완료")
print(f"사용 모델: gpt-4o-mini")
print(f"Temperature: 0 (일관된 답변)")

### 8단계: 체인 (Chain) 생성

이제 모든 구성 요소를 `|` 연산자로 연결하여 RAG 체인을 완성합니다.

**체인 흐름**:
```
1. 입력된 질문으로 문서 검색 (retriever)
2. 검색된 문서와 질문을 프롬프트에 삽입
3. LLM에 프롬프트 전달
4. LLM 응답을 문자열로 파싱
5. 최종 답변 반환
```

In [ ]:
# ---------------------------------------------------
# 단계 8: RAG 체인 생성 — retriever + prompt + llm 연결
# ---------------------------------------------------
# ============================================================
# TODO: LCEL 파이프라인으로 RAG 체인을 조립하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "RAG 체인 생성 완료" 출력
# ============================================================


## RAG 체인 실행

이제 완성된 RAG 체인을 사용하여 문서에 대한 질문에 답변해봅니다.

In [ ]:
# ---------------------------------------------------
# RAG 체인 실행 — 질문 1: 문서 주제 파악
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 질문에 답변하세요
# 힌트: rag_chain.invoke("이 문서의 주요 주제는 무엇인가요?")
# 예상 결과: 문서 기반 답변 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# RAG 체인 실행 — 질문 2: 구체적인 정보 질의
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 추진 과제에 대해 질문하세요
# 힌트: rag_chain.invoke("디지털 정부 혁신의 주요 추진 과제는 무엇인가요?")
# 예상 결과: 문서에서 찾은 추진 과제 답변 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# RAG 체인 실행 — 질문 3: 문서에 없는 내용 (답변 거절 테스트)
# ---------------------------------------------------
# ============================================================
# TODO: 문서에 없는 내용을 질문하여 모델의 답변 거절 능력을 테스트하세요
# 힌트: rag_chain.invoke("블록체인 기술의 미래 전망은 어떻게 되나요?")
# 예상 결과: "주어진 문서에서 해당 정보를 찾을 수 없습니다" 류의 답변
# ============================================================


## 전체 코드 (통합 버전)

위에서 단계별로 구현한 RAG 파이프라인을 하나의 코드로 정리했습니다.

In [ ]:
# ---------------------------------------------------
# 전체 RAG 파이프라인 통합 코드 (8단계 한 번에)
# ---------------------------------------------------
# ============================================================
# TODO: 위에서 단계별로 구현한 전체 RAG 파이프라인을 하나의 셀로 작성하세요
# 힌트: 1~8단계를 순서대로 — 로드 → 분할 → 임베딩 → 벡터저장 → 검색기 → 프롬프트 → LLM → 체인
# 예상 결과: 질문에 대한 답변 출력
# ============================================================


## 💡 핵심 정리

### RAG 파이프라인 8단계

1. **문서 로드**: PDF, 웹 등 다양한 형식의 문서를 읽어옴
2. **문서 분할**: 문서를 작은 청크로 나누어 검색 효율성 향상
3. **임베딩**: 텍스트를 벡터로 변환하여 의미 기반 검색 가능하게 함
4. **벡터 저장**: 임베딩된 청크를 벡터 DB에 저장
5. **검색기**: 질문과 관련된 문서를 자동으로 검색
6. **프롬프트**: 검색된 문서와 질문을 조합
7. **LLM**: 최종 답변 생성
8. **체인**: 전체 과정을 하나로 연결

### 주요 장점

- ✅ **정확성**: 외부 문서 기반으로 환각(Hallucination) 감소
- ✅ **최신성**: 문서만 업데이트하면 최신 정보 반영 가능
- ✅ **추적성**: 답변의 출처 문서 확인 가능
- ✅ **확장성**: 문서를 추가하여 지식 범위 확장 용이

### 개선 방향

다음 노트북에서는 RAG 성능을 향상시키는 고급 기법들을 다룹니다:
- Ensemble Retriever (여러 검색 방식 결합)
- Reranking (검색 결과 재정렬)
- Query Transformation (질문 개선)
- Contextual Compression (문맥 압축)

---

## 마무리 정리

### 8단계 RAG 파이프라인 요약

| 단계 | 구성 요소 | 핵심 역할 |
|------|-----------|-----------|
| 1. Load | `PyPDFLoader` | PDF에서 Document 객체 생성 |
| 2. Split | `RecursiveCharacterTextSplitter` | 청크 분할 (chunk_size/overlap) |
| 3. Embed | `OpenAIEmbeddings` | 텍스트 → 벡터 변환 |
| 4. Store | `FAISS` | 벡터 인덱스 저장 |
| 5. Retrieve | `.as_retriever()` | 쿼리와 유사한 문서 검색 |
| 6. Prompt | `ChatPromptTemplate` | 컨텍스트+질문 템플릿 결합 |
| 7. LLM | `ChatOpenAI` | 답변 생성 |
| 8. Chain | `|` (LCEL 파이프) | 전체 흐름 연결 |

### 핵심 파라미터 기준점

| 파라미터 | 기본값 | 조정 방향 |
|----------|--------|-----------|
| `chunk_size` | 1000 | 문서가 길면 늘리고, 정밀도가 필요하면 줄여요 |
| `chunk_overlap` | 200 | chunk_size의 10~20% 권장 |
| `k` (검색 문서 수) | 4 | 복잡한 질문엔 늘리고, 속도 중시엔 줄여요 |

### 다음 노트북 예고

**01-RAG-Web-Based** — PDF 대신 웹 페이지를 실시간으로 로드해 RAG를 구성하는 방법을 배워요. `WebBaseLoader`와 `BeautifulSoup`으로 Wikipedia 등 웹 문서를 바로 활용해요.